[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gaurav14cs17/Multimodal-Deep-Learning/blob/main/01_Multimodal_Foundations/02_modality_encoders.ipynb)

# 02. Modality Encoders: Image & Text

**This notebook covers:**
- How ViT (Vision Transformer) encodes images — with visualization
- How text encoders (BERT-style) produce embeddings
- Building both from scratch and using pretrained versions
- Visualizing what each encoder learns

---

In [ ]:
# ============================================================
#  Colab Setup (run this cell first if on Google Colab)
# ============================================================
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = "https://github.com/Gaurav14cs17/Multimodal-Deep-Learning.git"
    REPO_DIR = "/content/Multimodal-Deep-Learning"

    if not os.path.exists(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
        !pip install -q -r {REPO_DIR}/requirements.txt

    os.chdir(f"{REPO_DIR}/01_Multimodal_Foundations")
    os.makedirs(f"{REPO_DIR}/assets", exist_ok=True)
    print(f"Colab ready — working in {os.getcwd()}")
else:
    os.makedirs("../assets", exist_ok=True)

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from utils.visualization import *
from utils.helpers import count_parameters

set_style()

## Part 1: Vision Transformer (ViT) from Scratch

**Key idea:** Split an image into patches, treat each patch as a "token", then apply a transformer.

```
Image (224×224) → 16×16 patches → 196 patch tokens → Transformer → [CLS] embedding
```

In [ ]:
# STEP 1: Visualize patch extraction

def visualize_patches(image_tensor, patch_size=4):
    """Show how an image is split into patches."""
    C, H, W = image_tensor.shape
    n_patches_h = H // patch_size
    n_patches_w = W // patch_size
    n_patches = n_patches_h * n_patches_w

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # Original image with grid
    ax = axes[0]
    img_np = image_tensor.permute(1, 2, 0).numpy()
    img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    ax.imshow(img_np)
    for i in range(0, H + 1, patch_size):
        ax.axhline(y=i, color='red', linewidth=1, alpha=0.7)
    for j in range(0, W + 1, patch_size):
        ax.axvline(x=j, color='red', linewidth=1, alpha=0.7)
    ax.set_title(f'Image with {patch_size}×{patch_size} patch grid\n({n_patches} patches total)', fontsize=12)
    ax.axis('off')

    # Show individual patches
    ax = axes[1]
    show_n = min(n_patches, 16)
    grid_size = int(np.ceil(np.sqrt(show_n)))
    
    patches = image_tensor.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)
    patches = patches.contiguous().view(C, -1, patch_size, patch_size)
    patches = patches.permute(1, 2, 3, 0)  # [N, H, W, C]

    combined = np.zeros((grid_size * (patch_size + 1), grid_size * (patch_size + 1), 3))
    for idx in range(show_n):
        r, c = idx // grid_size, idx % grid_size
        patch = patches[idx].numpy()
        patch = (patch - patch.min()) / (patch.max() - patch.min() + 1e-8)
        y_start = r * (patch_size + 1)
        x_start = c * (patch_size + 1)
        combined[y_start:y_start+patch_size, x_start:x_start+patch_size] = patch

    ax.imshow(combined)
    ax.set_title(f'First {show_n} patches (extracted)', fontsize=12)
    ax.axis('off')

    plt.tight_layout()
    return fig

# Create a colorful test image
test_image = torch.zeros(3, 16, 16)
test_image[0, :8, :8] = 1.0    # Red top-left
test_image[1, :8, 8:] = 1.0    # Green top-right
test_image[2, 8:, :8] = 1.0    # Blue bottom-left
test_image[:, 8:, 8:] = 0.8     # White bottom-right
test_image += torch.randn_like(test_image) * 0.1

fig = visualize_patches(test_image, patch_size=4)
plt.show()

In [ ]:
# STEP 2: Build ViT from scratch

class PatchEmbedding(nn.Module):
    """Convert image into patch embeddings using a convolution."""
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=128):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, 
                             kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x: [B, C, H, W] -> [B, embed_dim, H/P, W/P] -> [B, N, embed_dim]
        x = self.proj(x)                    # [B, embed_dim, grid_h, grid_w]
        x = x.flatten(2).transpose(1, 2)    # [B, N_patches, embed_dim]
        return x


class ViTFromScratch(nn.Module):
    """Minimal Vision Transformer."""
    def __init__(self, img_size=32, patch_size=4, in_channels=3,
                 embed_dim=128, n_heads=4, n_layers=4, n_classes=10):
        super().__init__()
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        n_patches = (img_size // patch_size) ** 2

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches + 1, embed_dim) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, batch_first=True,
            dropout=0.1, activation='gelu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, n_classes)

    def forward(self, x, return_features=False):
        B = x.shape[0]
        x = self.patch_embed(x)                                # [B, N, D]
        cls = self.cls_token.expand(B, -1, -1)                 # [B, 1, D]
        x = torch.cat([cls, x], dim=1)                         # [B, N+1, D]
        x = x + self.pos_embed                                 # Add position info
        x = self.transformer(x)                                # [B, N+1, D]
        x = self.norm(x)
        cls_output = x[:, 0]                                   # [CLS] token
        
        if return_features:
            return cls_output
        return self.head(cls_output)


vit = ViTFromScratch(img_size=32, patch_size=4, embed_dim=128, n_layers=4)
count_parameters(vit)

# Test forward pass
dummy = torch.randn(2, 3, 32, 32)
out = vit(dummy)
print(f"\nInput shape:  {dummy.shape}")
print(f"Output shape: {out.shape}")

features = vit(dummy, return_features=True)
print(f"Feature shape: {features.shape}  ← This goes to the shared space")

In [ ]:
# STEP 3: Visualize what ViT sees at each stage

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('ViT Processing Pipeline (Visual)', fontsize=16, fontweight='bold')

img = torch.randn(1, 3, 32, 32)

# Stage 1: Original image
ax = axes[0, 0]
ax.imshow(img[0].permute(1, 2, 0).clamp(0, 1).numpy())
ax.set_title('1. Input Image\n(3 × 32 × 32)')
ax.axis('off')

# Stage 2: Patch embeddings
ax = axes[0, 1]
patch_emb = vit.patch_embed(img)  # [1, 64, 128]
ax.imshow(patch_emb[0].detach().numpy()[:, :32], aspect='auto', cmap='viridis')
ax.set_title(f'2. Patch Embeddings\n{patch_emb.shape[1]} patches × {patch_emb.shape[2]} dim')
ax.set_xlabel('Embedding dimensions (first 32)')
ax.set_ylabel('Patch index')

# Stage 3: Position embeddings
ax = axes[0, 2]
pos = vit.pos_embed[0].detach().numpy()
ax.imshow(pos[:, :32], aspect='auto', cmap='coolwarm')
ax.set_title(f'3. Position Embeddings\n{pos.shape[0]} positions (CLS + patches)')
ax.set_xlabel('Embedding dimensions (first 32)')
ax.set_ylabel('Position')

# Stage 4: After transformer (token representations)
ax = axes[1, 0]
with torch.no_grad():
    B = 1
    x = vit.patch_embed(img)
    cls = vit.cls_token.expand(B, -1, -1)
    x = torch.cat([cls, x], dim=1) + vit.pos_embed
    x = vit.transformer(x)

ax.imshow(x[0].numpy()[:, :32], aspect='auto', cmap='viridis')
ax.set_title(f'4. After Transformer\n{x.shape[1]} tokens × {x.shape[2]} dim')
ax.set_xlabel('Embedding dimensions (first 32)')
ax.set_ylabel('Token index (0=CLS)')

# Stage 5: CLS token (final feature)
ax = axes[1, 1]
cls_feat = x[0, 0].numpy()
ax.bar(range(len(cls_feat[:64])), cls_feat[:64], color='#9B59B6', alpha=0.7)
ax.set_title(f'5. [CLS] Token Vector\n(first 64 of {len(cls_feat)} dims)')
ax.set_xlabel('Dimension')
ax.set_ylabel('Value')

# Stage 6: Classification logits
ax = axes[1, 2]
logits = vit(img).detach().numpy()[0]
bars = ax.bar(range(10), logits, color='#E74C3C', alpha=0.7)
ax.set_title('6. Output Logits\n(10 classes)')
ax.set_xlabel('Class')
ax.set_ylabel('Logit value')

plt.tight_layout()
plt.savefig('../assets/vit_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 2: Text Encoder

For multimodal models, the text encoder converts a sentence into a fixed-size vector.  
Common approach: Transformer encoder → take [CLS] token → project to shared dim.

In [ ]:
class TextEncoderFromScratch(nn.Module):
    """BERT-style text encoder with visualization hooks."""
    def __init__(self, vocab_size=30000, embed_dim=128, n_heads=4,
                 n_layers=2, max_len=64):
        super().__init__()
        self.token_embed = nn.Embedding(vocab_size, embed_dim)
        self.pos_embed = nn.Embedding(max_len, embed_dim)
        self.type_embed = nn.Embedding(2, embed_dim)  # segment A/B

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(embed_dim)
        self._attention_weights = []

    def forward(self, input_ids, token_type_ids=None):
        B, T = input_ids.shape
        positions = torch.arange(T, device=input_ids.device).unsqueeze(0).expand(B, -1)

        x = self.token_embed(input_ids) + self.pos_embed(positions)
        if token_type_ids is not None:
            x = x + self.type_embed(token_type_ids)

        x = self.transformer(x)
        x = self.norm(x)
        return x[:, 0]  # [CLS] representation


text_enc = TextEncoderFromScratch(vocab_size=30000, embed_dim=128)
count_parameters(text_enc)

# Test
dummy_ids = torch.randint(0, 30000, (2, 16))
output = text_enc(dummy_ids)
print(f"\nInput shape:  {dummy_ids.shape}  (batch=2, seq_len=16)")
print(f"Output shape: {output.shape}  (batch=2, embed_dim=128)")

In [ ]:
# Visualize text encoding pipeline

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle('Text Encoding Pipeline', fontsize=16, fontweight='bold')

tokens = ['[CLS]', 'a', 'photo', 'of', 'a', 'cute', 'cat', '[SEP]', '[PAD]', '[PAD]']
dummy_ids = torch.randint(0, 100, (1, len(tokens)))

# 1. Token IDs
ax = axes[0]
ax.barh(range(len(tokens)), dummy_ids[0].numpy(), color='#3498DB', alpha=0.7)
ax.set_yticks(range(len(tokens)))
ax.set_yticklabels(tokens)
ax.set_title('1. Token IDs')
ax.set_xlabel('ID value')
ax.invert_yaxis()

# 2. Token embeddings
ax = axes[1]
tok_emb = text_enc.token_embed(dummy_ids)[0].detach().numpy()
ax.imshow(tok_emb[:, :32], aspect='auto', cmap='RdBu_r')
ax.set_yticks(range(len(tokens)))
ax.set_yticklabels(tokens)
ax.set_title('2. Token Embeddings')
ax.set_xlabel('Dim (first 32)')

# 3. + Position embeddings
ax = axes[2]
pos_emb = text_enc.pos_embed(torch.arange(len(tokens)))
combined = tok_emb + pos_emb.detach().numpy()
ax.imshow(combined[:, :32], aspect='auto', cmap='RdBu_r')
ax.set_yticks(range(len(tokens)))
ax.set_yticklabels(tokens)
ax.set_title('3. Token + Position')
ax.set_xlabel('Dim (first 32)')

# 4. Output [CLS] vector
ax = axes[3]
with torch.no_grad():
    output = text_enc(dummy_ids)
ax.barh(range(32), output[0, :32].numpy(), color='#2ECC71', alpha=0.7)
ax.set_title(f'4. [CLS] Output\n(first 32 of {output.shape[1]})')
ax.set_xlabel('Value')
ax.set_ylabel('Dimension')

plt.tight_layout()
plt.savefig('../assets/text_encoding_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()

## Part 3: Using Pretrained Encoders (Practical)

In practice, you use pretrained encoders and just add a projection head.

In [ ]:
from transformers import AutoModel, AutoTokenizer, ViTModel, ViTFeatureExtractor
from torchvision import models

# Option 1: Use a pretrained ResNet as image encoder
resnet = models.resnet18(weights=None)  # no download needed for structure
resnet_features = nn.Sequential(*list(resnet.children())[:-1])  # remove classifier

dummy_img = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    feat = resnet_features(dummy_img)
print(f"ResNet18 feature shape: {feat.shape}  → flatten to {feat.flatten(1).shape}")
print(f"Then project: Linear(512, shared_dim)")

# Count params
print(f"\nResNet18 total params: {sum(p.numel() for p in resnet.parameters()):,}")

In [ ]:
# Comparison of encoder sizes (for low-compute planning)

encoder_data = {
    'Model': ['ResNet-18', 'ResNet-50', 'ViT-Tiny', 'ViT-Small', 'ViT-Base',
              'BERT-Tiny', 'BERT-Mini', 'BERT-Base', 'DistilBERT'],
    'Params (M)': [11.7, 25.6, 5.7, 22.1, 86.6,
                   4.4, 11.3, 110, 66],
    'Type': ['Vision', 'Vision', 'Vision', 'Vision', 'Vision',
             'Text', 'Text', 'Text', 'Text'],
    'Low Compute': ['Yes', 'Yes', 'Yes', 'Yes', 'Moderate',
                      'Yes', 'Yes', 'Moderate', 'Yes']
}

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#E74C3C' if t == 'Vision' else '#3498DB' for t in encoder_data['Type']]
edge_colors = ['green' if lc == 'Yes' else 'orange' for lc in encoder_data['Low Compute']]

bars = ax.barh(encoder_data['Model'], encoder_data['Params (M)'], 
               color=colors, alpha=0.7, edgecolor=edge_colors, linewidth=2)

for bar, params in zip(bars, encoder_data['Params (M)']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{params}M', va='center', fontweight='bold')

ax.set_xlabel('Parameters (Millions)', fontsize=12)
ax.set_title('Encoder Size Comparison\n(green border = low-compute friendly)', fontsize=14, fontweight='bold')

import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='#E74C3C', alpha=0.7, label='Vision Encoder'),
    mpatches.Patch(color='#3498DB', alpha=0.7, label='Text Encoder'),
], fontsize=11)

plt.tight_layout()
plt.savefig('../assets/encoder_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Takeaways

1. **ViT** splits images into patches → treats them as tokens → transformer processes them
2. **Text encoder** converts tokens → embeddings → transformer → [CLS] output
3. Both produce a **fixed-size vector** that goes into the shared space
4. For low compute: use **ViT-Tiny/Small** and **DistilBERT/BERT-Mini**
5. The **projection head** maps encoder outputs to the shared dimension

---
**Next:** `03_fusion_strategies.ipynb` - How to combine modalities